In [13]:
# Sel ini menghasilkan TIGA dataset cabang (kota) terpisah untuk Tugas Mandiri Pertemuan 3
import numpy as np
import pandas as pd

kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
tanggal_range = pd.date_range("2026-08-01", "2026-08-31", freq="D")

cabang_kota = {"Magelang": 101, "Yogyakarta": 202, "Semarang": 303}

for kota, seed in cabang_kota.items():
    np.random.seed(seed)  # seed berbeda tiap kota agar datanya bervariasi, namun tetap konsisten/reproducible
    n = 200
    data_cabang = {
        "order_id": [f"{kota[:3].upper()}-{2000 + i}" for i in range(n)],
        "tanggal": np.random.choice(tanggal_range, size=n),
        "kategori": np.random.choice(kategori_list, size=n, p=[0.25, 0.25, 0.20, 0.15, 0.15]),
        "unit_terjual": np.random.randint(1, 8, size=n),
        "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000, 250000], size=n),
        "metode_pembayaran": np.random.choice(metode_bayar_list, size=n),
    }
    df_cabang = pd.DataFrame(data_cabang)
    df_cabang["kota"] = kota
    nama_file = f"transaksi_{kota.lower()}.csv"
    df_cabang.to_csv(nama_file, index=False)
    print(f"Berkas '{nama_file}' berhasil dibuat: {df_cabang.shape[0]} baris")

print("\nKetiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.")

Berkas 'transaksi_magelang.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_yogyakarta.csv' berhasil dibuat: 200 baris
Berkas 'transaksi_semarang.csv' berhasil dibuat: 200 baris

Ketiga berkas CSV cabang siap digunakan untuk Tugas Mandiri.


In [14]:
# Ganti 'albani' jika username HDFS Anda berbeda
!hdfs dfs -mkdir -p /user/albani/ecommerce/raw
!hdfs dfs -mkdir -p /user/albani/ecommerce/processed

# Menampilkan struktur direktori yang telah dibuat
!hdfs dfs -ls -R /user/albani/ecommerce

drwxr-xr-x   - albani supergroup          0 2026-09-28 09:57 /user/albani/ecommerce/processed
drwxr-xr-x   - albani supergroup          0 2026-09-28 09:57 /user/albani/ecommerce/raw


In [20]:
# Mengunggah berkas CSV ke direktori raw di HDFS
!hdfs dfs -put transaksi_magelang.csv /user/albani/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/albani/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/albani/ecommerce/raw/

# Menampilkan isi direktori raw beserta ukuran berkasnya
!hdfs dfs -ls -h /user/albani/ecommerce/raw

put: `/user/albani/ecommerce/raw/transaksi_magelang.csv': File exists
put: `/user/albani/ecommerce/raw/transaksi_yogyakarta.csv': File exists
put: `/user/albani/ecommerce/raw/transaksi_semarang.csv': File exists
Found 3 items
-rw-r--r--   1 albani supergroup     12.0 K 2026-09-28 09:59 /user/albani/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 albani supergroup     11.9 K 2026-09-28 09:59 /user/albani/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 albani supergroup     12.4 K 2026-09-28 09:59 /user/albani/ecommerce/raw/transaksi_yogyakarta.csv


In [21]:
from io import StringIO
import subprocess
import pandas as pd


# Fungsi bantuan untuk membaca file dari HDFS langsung ke DataFrame Pandas
def read_hdfs_csv(hdfs_path):
  result = subprocess.run(
      ["hdfs", "dfs", "-cat", hdfs_path], capture_output=True, text=True
  )
  return pd.read_csv(StringIO(result.stdout))


# Membaca ketiga file dari HDFS
df_magelang = read_hdfs_csv("/user/albani/ecommerce/raw/transaksi_magelang.csv")
df_yogyakarta = read_hdfs_csv(
    "/user/albani/ecommerce/raw/transaksi_yogyakarta.csv"
)
df_semarang = read_hdfs_csv("/user/albani/ecommerce/raw/transaksi_semarang.csv")

# Menggabungkan ketiga DataFrame
df_gabungan = pd.concat(
    [df_magelang, df_yogyakarta, df_semarang], ignore_index=True
)

# Membuktikan hasil gabungan berisi transaksi dari ketiga kota
print("Banyaknya transaksi per kota:")
print(df_gabungan["kota"].value_counts())
print("\nUkuran total DataFrame gabungan:", df_gabungan.shape)
df_gabungan.head()

Banyaknya transaksi per kota:
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

Ukuran total DataFrame gabungan: (600, 7)


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [22]:
# 1. Menambahkan kolom total_pendapatan (unit_terjual x harga_satuan)
df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
)

# 2. Membuat tabel ringkasan (groupby) total pendapatan per kota dan per kategori
df_ringkasan = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
)

# Displays
print("--- Ringkasan Total Pendapatan per Kota & Kategori ---")
print(df_ringkasan.head(10))

# 3. Menyimpan kedua hasil sebagai CSV lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)

--- Ringkasan Total Pendapatan per Kota & Kategori ---
       kota                kategori  total_pendapatan
0  Magelang              Elektronik          18775000
1  Magelang                 Fashion          27750000
2  Magelang  Kesehatan & Kecantikan          17375000
3  Magelang       Makanan & Minuman          17525000
4  Magelang            Rumah Tangga          12200000
5  Semarang              Elektronik          21425000
6  Semarang                 Fashion          26425000
7  Semarang  Kesehatan & Kecantikan          13525000
8  Semarang       Makanan & Minuman          19750000
9  Semarang            Rumah Tangga          10975000


In [23]:
# Mengunggah berkas olahan ke direktori processed di HDFS
!hdfs dfs -put -f data_gabungan_bersih.csv /user/albani/ecommerce/processed/
!hdfs dfs -put -f ringkasan_kota_kategori.csv /user/albani/ecommerce/processed/

# Verifikasi isi direktori processed
!hdfs dfs -ls -h /user/albani/ecommerce/processed

Found 2 items
-rw-r--r--   1 albani supergroup     40.3 K 2026-09-28 10:02 /user/albani/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 albani supergroup        530 2026-09-28 10:02 /user/albani/ecommerce/processed/ringkasan_kota_kategori.csv
